# DE2 - Assignment 2: Text - Inverted Index
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** *B - données de capteurs urbains*

**Names:** *Rabahi Enzo - Couzinet Lorenzo*

Complete the cells below. Refer to `DE2_Lab2_Overview_EN.md` and `helper_assignment2-de2_esiee.md` for details.

## 0. Setup

In [1]:
import os
import io
import csv
import sys
import time
import pathlib
from datetime import datetime
from urllib.parse import urlparse

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

DE2_SPARK_DRIVER_HOST  = os.environ.get("DE2_SPARK_DRIVER_HOST",  "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url  = spark_session.sparkContext.uiWebUrl
    ui_port = urlparse(ui_url).port or 4040
    print("Spark version:", spark_session.version)
    print("Spark UI:", ui_url)
    print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")


spark = (
    SparkSession.builder
    .appName("de2-assignment2")
    .master("local[*]")
    .config("spark.driver.host",        DE2_SPARK_DRIVER_HOST)
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.ui.bindAddress",     DE2_SPARK_BIND_ADDRESS)
    .getOrCreate()
)

show_spark_ui(spark)

# Output directories - - created once here, reused in steps 4-7
PARQUET_PATH = "outputs/lab2/inverted_index"
CSV_PATH     = "outputs/lab2/inverted_index_csv"
PROOF_DIR    = pathlib.Path("proof")
METRICS_PATH = "lab2_metrics_log.csv"   # project root, NOT inside proof/

for p in [PARQUET_PATH, CSV_PATH]:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)
PROOF_DIR.mkdir(parents=True, exist_ok=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 16:37:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 1. Corpus Ingestion
Load your track's text corpus with an explicit schema. Print row count and sample.

In [ ]:
# Explicit schema 
schema = StructType([
    StructField("doc_id", StringType(), nullable=False),
    StructField("text",   StringType(), nullable=True),
])

CORPUS_PATH = "../data/smart_mobility_dataset.csv"   

corpus_df = (
    spark.read
    .option("header",    "true")
    .option("quote",     '"')
    .option("escape",    '"')
    .option("multiLine", "true")   
    .schema(schema)
    .csv(CORPUS_PATH)
    .filter(F.col("doc_id").isNotNull() & F.col("text").isNotNull())
)

corpus_df.cache()

row_count = corpus_df.count()
print(f"Total documents loaded : {row_count:,}")
corpus_df.show(5, truncate=80)

26/05/11 16:37:39 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 15, schema size: 2
CSV file: file:///Users/lorenzo/Documents/Cours/E4FDS2/Data%20Engineering%202/data/smart_mobility_dataset.csv


Total documents loaded : 5,000
+-------------------+------------------+
|             doc_id|              text|
+-------------------+------------------+
|2024-03-01 00:00:00|40.842275292891834|
|2024-03-01 00:05:00|  40.8311193987152|
|2024-03-01 00:10:00| 40.81954876392327|
|2024-03-01 00:15:00| 40.72584887921568|
|2024-03-01 00:20:00| 40.81326467769622|
+-------------------+------------------+
only showing top 5 rows


## 2. Text Normalization
Lowercase, tokenize, remove stop-words. Print token counts before/after.

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "if", "in", "on", "at", "to",
    "for", "of", "with", "by", "from", "is", "it", "its", "be", "was",
    "are", "were", "been", "has", "have", "had", "do", "does", "did",
    "not", "no", "so", "as", "this", "that", "these", "those", "i",
    "we", "you", "he", "she", "they", "his", "her", "our", "their",
    "will", "would", "can", "could", "may", "might", "shall", "should",
    "than", "then", "there", "also", "all", "any", "each", "more",
    "into", "about", "up", "out", "what", "which", "who", "whom", "how",
    "when", "where", "why", "s", "t", "re", "ll", "ve", "don"
}

stop_words_list = list(STOP_WORDS)  

tokenized_df = corpus_df.withColumn(
    "tokens_raw",
    F.split(F.lower(F.col("text")), r"[\s\W]+")
)

raw_tokens_df = tokenized_df.withColumn("token", F.explode("tokens_raw"))

tokens_before = raw_tokens_df.filter(F.col("token") != "").count()
print(f"Token count BEFORE stop-word removal : {tokens_before:,}")

tokens_df = (
    raw_tokens_df
    .filter(
        (F.col("token") != "") &
        (F.length("token") > 1) &
        (~F.col("token").isin(stop_words_list))
    )
    .select("doc_id", "token")
)

tokens_after = tokens_df.count()
reduction    = (1 - tokens_after / tokens_before) * 100
print(f"Token count AFTER  stop-word removal : {tokens_after:,}")
print(f"Reduction                            : {reduction:.1f} %")

tokens_df.show(10, truncate=False)

Token count BEFORE stop-word removal : 10,000
Token count AFTER  stop-word removal : 10,000
Reduction                            : 0.0 %
+-------------------+---------------+
|doc_id             |token          |
+-------------------+---------------+
|2024-03-01 00:00:00|40             |
|2024-03-01 00:00:00|842275292891834|
|2024-03-01 00:05:00|40             |
|2024-03-01 00:05:00|8311193987152  |
|2024-03-01 00:10:00|40             |
|2024-03-01 00:10:00|81954876392327 |
|2024-03-01 00:15:00|40             |
|2024-03-01 00:15:00|72584887921568 |
|2024-03-01 00:20:00|40             |
|2024-03-01 00:20:00|81326467769622 |
+-------------------+---------------+
only showing top 10 rows


## 3. Build Inverted Index
`groupBy(token).agg(collect_list(doc_id), count(*))`. Show sample and unique term count.

In [ ]:
pairs_df = tokens_df.dropDuplicates(["doc_id", "token"])

inverted_index_df = (
    pairs_df
    .groupBy("token")
    .agg(
        F.sort_array(F.collect_list("doc_id")).alias("doc_ids"),
        F.count("doc_id").alias("freq") 
    )
    .orderBy(F.col("freq").desc()) 
)

inverted_index_df.cache()

unique_terms = inverted_index_df.count()
print(f"Unique terms in index : {unique_terms:,}")

inverted_index_df.printSchema()
inverted_index_df.show(20, truncate=60)

Unique terms in index : 5,001
root
 |-- token: string (nullable = false)
 |-- doc_ids: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- freq: long (nullable = false)

+---------------+------------------------------------------------------------+----+
|          token|                                                     doc_ids|freq|
+---------------+------------------------------------------------------------+----+
|             40|[2024-03-01 00:00:00, 2024-03-01 00:05:00, 2024-03-01 00:...|5000|
| 60001638114008|                                       [2024-03-05 15:25:00]|   1|
| 60009183670379|                                       [2024-03-01 10:00:00]|   1|
| 60030035090456|                                       [2024-03-16 23:15:00]|   1|
| 60036588756023|                                       [2024-03-11 23:35:00]|   1|
|600396683877605|                                       [2024-03-08 16:35:00]|   1|
| 60056248865906|                              

## 4. Write to Parquet & CSV
Write index to `outputs/lab2/inverted_index/` (Parquet) and `outputs/lab2/inverted_index_csv/` (CSV).

In [ ]:
t0 = time.time()
(
    inverted_index_df
    .write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)
parquet_write_s = time.time() - t0
print(f"Parquet written in {parquet_write_s:.2f} s  ->  {PARQUET_PATH}")

index_for_csv = inverted_index_df.withColumn(
    "doc_ids", F.array_join("doc_ids", "|")
)

t0 = time.time()
(
    index_for_csv
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(CSV_PATH)
)
csv_write_s = time.time() - t0
print(f"CSV     written in {csv_write_s:.2f} s  ->  {CSV_PATH}")

print("\nSanity read from Parquet (5 rows):")
spark.read.parquet(PARQUET_PATH).show(5, truncate=60)

Parquet written in 0.53 s  ->  outputs/lab2/inverted_index
CSV     written in 0.12 s  ->  outputs/lab2/inverted_index_csv

Sanity read from Parquet (5 rows):
+--------------+------------------------------------------------------------+----+
|         token|                                                     doc_ids|freq|
+--------------+------------------------------------------------------------+----+
|            40|[2024-03-01 00:00:00, 2024-03-01 00:05:00, 2024-03-01 00:...|5000|
|60001638114008|                                       [2024-03-05 15:25:00]|   1|
|60009183670379|                                       [2024-03-01 10:00:00]|   1|
|60030035090456|                                       [2024-03-16 23:15:00]|   1|
|60036588756023|                                       [2024-03-11 23:35:00]|   1|
+--------------+------------------------------------------------------------+----+
only showing top 5 rows


## 5. Query Latency
Look up at least 3 terms. Record wall-clock latency for each. Save query plan.

In [ ]:
index_parquet = spark.read.parquet(PARQUET_PATH)

index_csv = (
    spark.read
    .option("header", "true")
    .csv(CSV_PATH)
    .withColumn("freq", F.col("freq").cast("long"))
)

QUERY_TERMS = ["data", "network", "algorithm"]   

latency_records = []

print(f"{'Term':<15} {'Format':<10} {'freq':>6}  {'latency_ms':>12}")
print("-" * 52)

for term in QUERY_TERMS:
    for fmt, df in [("parquet", index_parquet), ("csv", index_csv)]:
        t_start = time.perf_counter()
        result  = df.filter(F.col("token") == term).collect()
        elapsed = (time.perf_counter() - t_start) * 1_000   # ms

        freq = int(result[0]["freq"]) if result else 0
        print(f"{term:<15} {fmt:<10} {freq:>6}  {elapsed:>10.1f} ms")
        latency_records.append({
            "term":       term,
            "format":     fmt,
            "freq":       freq,
            "latency_ms": round(elapsed, 2),
        })

# Save proof/plan_query.txt
plan_query_path = PROOF_DIR / "plan_query.txt"

buf = io.StringIO()
old_stdout, sys.stdout = sys.stdout, buf
index_parquet.filter(F.col("token") == QUERY_TERMS[0]).explain("formatted")
sys.stdout = old_stdout
plan_query_path.write_text(buf.getvalue(), encoding="utf-8")
print(f"\nQuery plan saved  ->  {plan_query_path}")

print("\nFormatted query plan (Parquet, first term)")
index_parquet.filter(F.col("token") == QUERY_TERMS[0]).explain("formatted")

Term            Format       freq    latency_ms
----------------------------------------------------
data            parquet         0        54.5 ms
data            csv             0        58.0 ms
network         parquet         0        28.0 ms
network         csv             0        35.0 ms
algorithm       parquet         0        27.0 ms
algorithm       csv             0        37.4 ms

Query plan saved  ->  proof/plan_query.txt

Formatted query plan (Parquet, first term)
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [token#660, doc_ids#661, freq#662L]
Batched: true
Location: InMemoryFileIndex [file:/Users/lorenzo/Documents/Cours/E4FDS2/Data Engineering 2/lab2/outputs/lab2/inverted_index]
PushedFilters: [IsNotNull(token), EqualTo(token,data)]
ReadSchema: struct<token:string,doc_ids:array<string>,freq:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [token#660, doc_ids#661, freq#662L]

(3) Filter [codegen id :

## 6. Footprint Comparison
Compare Parquet vs CSV sizes. Record difference.

In [ ]:
def dir_size_bytes(path: str) -> int:
    """Recursively sum file sizes under `path`."""
    total = 0
    for root, _, files in os.walk(path):
        for fname in files:
            fp = os.path.join(root, fname)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total

def human_size(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

parquet_bytes = dir_size_bytes(PARQUET_PATH)
csv_bytes     = dir_size_bytes(CSV_PATH)
ratio         = csv_bytes / parquet_bytes if parquet_bytes else float("inf")

print("Storage footprint comparison")
print("=" * 48)
print(f"  Parquet : {human_size(parquet_bytes):>10}  ({parquet_bytes:,} bytes)")
print(f"  CSV     : {human_size(csv_bytes):>10}  ({csv_bytes:,} bytes)")
print(f"  Ratio   : CSV is {ratio:.2f}× larger than Parquet")
print(f"  Saving  : {human_size(abs(csv_bytes - parquet_bytes))} saved with Parquet")

footprint_info = {
    "parquet_bytes": parquet_bytes,
    "csv_bytes":     csv_bytes,
    "ratio":         round(ratio, 3),
}

Storage footprint comparison
  Parquet :    98.3 KB  (100,706 bytes)
  CSV     :   281.6 KB  (288,359 bytes)
  Ratio   : CSV is 2.86× larger than Parquet
  Saving  : 183.3 KB saved with Parquet


## 7. Evidence & Metrics
Save plans to `proof/`, fill `lab2_metrics_log.csv`, capture Spark UI screenshots.

In [ ]:
# proof/plan_index_build.txt 
build_plan_path = PROOF_DIR / "plan_index_build.txt"

buf = io.StringIO()
old_stdout, sys.stdout = sys.stdout, buf
inverted_index_df.explain("formatted")
sys.stdout = old_stdout
build_plan_path.write_text(buf.getvalue(), encoding="utf-8")
print(f"Index-build plan saved  ->  {build_plan_path}")

print("\n--- Physical Plan (inverted index build) ---")
inverted_index_df.explain("formatted")

# Confirm plan_query.txt exists 
plan_query_path = PROOF_DIR / "plan_query.txt"
assert plan_query_path.exists(), "plan_query.txt missing - re-run section 5!"
print(f"Query plan confirmed    ->  {plan_query_path}")

# lab2_metrics_log.csv 
# Columns: timestamp | step | metric | value | unit
metrics_rows = [
    ["1_ingestion",     "row_count",            row_count,                 "documents"],
    ["2_normalization", "tokens_before_filter",  tokens_before,             "tokens"],
    ["2_normalization", "tokens_after_filter",   tokens_after,              "tokens"],
    ["2_normalization", "reduction_pct",         round(reduction, 2),       "percent"],
    ["3_index_build",   "unique_terms",           unique_terms,              "terms"],
    ["4_write",         "parquet_write_s",        round(parquet_write_s, 3), "seconds"],
    ["4_write",         "csv_write_s",            round(csv_write_s, 3),     "seconds"],
    ["6_footprint",     "parquet_bytes",          parquet_bytes,             "bytes"],
    ["6_footprint",     "csv_bytes",              csv_bytes,                 "bytes"],
    ["6_footprint",     "csv_parquet_ratio",      footprint_info["ratio"],   "x"],
] + [
    ["5_query",
     f"latency_{r['term']}_{r['format']}",
     r["latency_ms"],
     "ms"]
    for r in latency_records
]

ts = datetime.now().isoformat(timespec="seconds")
with open(METRICS_PATH, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["timestamp", "step", "metric", "value", "unit"])
    for row in metrics_rows:
        writer.writerow([ts] + row)

print(f"\nMetrics log written  ->  {METRICS_PATH}")

ui_port = urlparse(spark.sparkContext.uiWebUrl).port or 4040

# List proof/ contents
print("\nFiles in proof/:")
for f in sorted(PROOF_DIR.iterdir()):
    print(f"  {f.name:<40}  {f.stat().st_size:>10,} bytes")

Index-build plan saved  ->  proof/plan_index_build.txt

--- Physical Plan (inverted index build) ---
== Physical Plan ==
AdaptiveSparkPlan (39)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (38)
               +- == Final Plan ==
                  ResultQueryStage (25)
                  +- * Sort (24)
                     +- ShuffleQueryStage (23), Statistics(sizeInBytes=625.7 KiB, rowCount=5.00E+3)
                        +- Exchange (22)
                           +- ObjectHashAggregate (21)
                              +- AQEShuffleRead (20)
                                 +- ShuffleQueryStage (19), Statistics(sizeInBytes=703.8 KiB, rowCount=5.00E+3)
                                    +- Exchange (18)
                                       +- ObjectHashAggregate (17)
                                          +- * HashAggregate (16)
                                             +- AQEShuffleRead (15)
                                        

## 8. Cleanup

In [ ]:
spark.stop()
print("Done.")

Done.
